In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from dask.distributed import wait

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

# BARRA-C2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"


In [2]:
client = Client(n_workers=24,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 24
Total threads: 24,Total memory: 111.76 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40675,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39317,Total threads: 1
Dashboard: /proxy/33699/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:39775,


Thesse define whether which sample we're calcualting and which model we're using.

In [3]:
sample = 'baseline'
reanalysis = 'BARRA-C2'

In [4]:
if sample == 'heatwave':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")
    mode_str = 'heatwave'
    
elif sample == 'baseline':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/no_hw_alpine_cluster.csv")
    mode_str = 'baseline'

if reanalysis == 'BARRA-C2':
    extent = [147, 152, -38.5, -32]
elif reanalysis == 'BARRA-R2':
    extent = None

lon_min, lon_max, lat_min, lat_max = extent

In [5]:
# Statistically significant w ssmin=20, and Mann-Whitney 
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'CRURWF1',
            'WOODLWN1',
            'BOCORWF1',
            'BODWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [6]:
def get_days(days, nc_dir,
             tz='Australia/Brisbane'):
    
    local_dates = pd.DatetimeIndex(pd.to_datetime(days['date'])).tz_localize(tz)
    utc_start = local_dates.min().tz_convert('UTC').normalize()
    utc_end   = (local_dates.max() + pd.Timedelta(days=1)).tz_convert('UTC').normalize()

    # Months from UTC window
    utc_months = pd.date_range(utc_start, utc_end - pd.Timedelta(seconds=1),
                               freq='MS', tz='UTC').strftime('%Y%m')
    year_month_filter = set(utc_months)

    selected_files = [
        os.path.join(nc_dir, f)
        for f in os.listdir(nc_dir)
        if any(ym in f for ym in year_month_filter)
    ]

    ds = xr.open_mfdataset(
        selected_files,
        combine='by_coords',
        parallel=True,
        chunks="auto"
    )

    ds_sub = ds.sel(time=slice(utc_start.tz_localize(None),
                               utc_end.tz_localize(None)))

    times_local = ds_sub.indexes['time'].tz_localize('UTC').tz_convert(tz)
    mask = times_local.normalize().isin(local_dates.normalize())
    ds_sub = ds_sub.isel(time=mask)
    ds_sub = ds_sub.assign_coords(local_hour=('time', times_local[mask].hour))
    
    return ds_sub

In [7]:
def get_ds():
    u_hw_cluster = get_days(cluster_dates, u_path)
    v_hw_cluster = get_days(cluster_dates, v_path)
    
    ds = xr.merge([u_hw_cluster, v_hw_cluster])

    ds = ds.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
    })
    
    return ds


In [8]:
ds_subset = get_ds().persist()

ValueError: extent must be provided: [lon_min, lon_max, lat_min, lat_max]

In [ ]:
ds_subset

In [ ]:
# This computes the composite of variance
ds_subset['windspeed'] = np.sqrt(ds_subset['ua100m']**2 + ds_subset['va100m']**2)

def temporal_variance(x):
    # x has dimensions (time, lat, lon)
    return x.var(dim="time", ddof=1)  # use ddof=1 for unbiased sample variance

hourly_var_per_point = ds_subset['windspeed'].groupby("time.hour").map(temporal_variance)

In [ ]:
# This computes the hourly u v field composite.
hourly_composite =  ds_subset.groupby("time.hour").mean()

In [ ]:
# This computes the composites of minimum, maximum, and diunal amplitude
def compute_windspeed_composites(windspeed):
    """
    Compute windspeed composites (max, min, amplitude) from u and v.
    
    Parameters
    ----------
    u, v : xarray.DataArray
        Wind vector components (time x lat x lon)
    
    Returns
    -------
    dict
        {"max": DataArray, "min": DataArray, "diff": DataArray}
    """
    # Compute windspeed
    
    # Compute composites along time dimension
    composite_max = windspeed.max(dim="time").compute()
    composite_min = windspeed.min(dim="time").compute()
    diurnal_amp = (composite_max - composite_min).compute()
    
    return composite_max, composite_min, diurnal_amp

max_speed, min_speed, diurnal_amp = compute_windspeed_composites(ds_subset['windspeed'])

In [ ]:
# --- Assemble into one Dataset ---
results = xr.Dataset(
    {
        "windspeed_variance": hourly_var_per_point,
        "windspeed_mean": hourly_composite["windspeed"],
        "u_mean": hourly_composite["ua100m"],
        "v_mean": hourly_composite["va100m"],
        "windspeed_max": max_speed,
        "windspeed_min": min_speed,
        "diurnal_amp": diurnal_amp,
    }
)

results = results.assign_attrs(reanalysis=reanalysis, mode=mode_str)

# Write to NetCDF lazily with dask
with ProgressBar():
    results.to_netcdf(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc", compute=True)

In [ ]:
xr.open_dataset(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc")